# Final FNO model: 1D Burgers (PK-2026-011)
Trains **one** Fourier Neural Operator on **all 48 training simulations** (240 examples) at 128 points.

**Settings:** `modes = 64`, 500 epochs, width 64, 4 Fourier layers, seed 0.
**Training recipe:** Adam, learning rate 1e-3 with cosine decay, weight decay 1e-4, batch size 16, relative L2 loss.
**Output:** the saved model `fno_modes64_all48_seed0.pt` (weights from the last epoch).

The model learns one step: given the solution at one time (128 numbers), predict the solution at the next time step (128 numbers).

## 1. Setup

In [1]:
import numpy as np
import torch
import torch.nn as nn

def set_seed(seed):
    """Fix the randomness so the run can be repeated."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# Folder that contains train.npz (change if needed)
DATA_DIR = "/Users/vedbhatt/Desktop/AML/learning-physical-dynamics-across-resolutions-main 6.08.58 PM/data"

# Settings
MODES    = 64      # Fourier modes kept in each Fourier layer
WIDTH    = 64      # channels inside the network
N_LAYERS = 4       # number of Fourier layers
EPOCHS   = 500
SEED     = 0

Running on: cpu


## 2. Load the training data 


In [3]:
data = np.load(f"/Users/vedbhatt/Desktop/AML/learning-physical-dynamics-across-resolutions-main 6.08.58 PM/data/train.npz")

X = torch.tensor(data["field"],  dtype=torch.float32, device=device)   # input:  solution now        (240, 128)
Y = torch.tensor(data["target"], dtype=torch.float32, device=device)   # target: solution next step (240, 128)

print("inputs :", tuple(X.shape))
print("targets:", tuple(Y.shape))

inputs : (240, 128)
targets: (240, 128)


## 3. Network input: the value and the position of each grid point

In [4]:
def make_input(u):
    """(batch, N) -> (batch, N, 2): the value u and its position x in [0, 1)."""
    batch, n_points = u.shape
    x = torch.linspace(0, 1, n_points + 1, device=u.device)[:-1]
    x = x.expand(batch, n_points)
    return torch.stack([u, x], dim=-1)

## 4. Fourier layer
Converts the signal to waves, keeps the lowest `modes` waves multiplied by learned weights, drops the rest, and converts back.
The weights belong to waves, not to grid points, so the same layer works on any grid size.

In [5]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.modes = modes
        scale = 1 / (in_channels * out_channels)
        self.weights = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x):                                   # x: (batch, channels, n_points)
        batch, _, n_points = x.shape
        x_freq = torch.fft.rfft(x)                          # signal -> waves
        out_freq = torch.zeros(batch, self.weights.shape[1], n_points // 2 + 1,
                               dtype=torch.cfloat, device=x.device)
        k = min(self.modes, n_points // 2 + 1)
        out_freq[:, :, :k] = torch.einsum("bix,iox->box",   # lowest k waves x learned weights
                                          x_freq[:, :, :k], self.weights[:, :, :k])
        return torch.fft.irfft(out_freq, n=n_points)        # waves -> signal

## 5. The FNO
Lift to `WIDTH` channels, apply 4 Fourier layers (Fourier path + local path, then GELU), project back to one value per grid point.

In [6]:
class FNO1d(nn.Module):
    def __init__(self, modes=64, width=64, n_layers=4):
        super().__init__()
        self.lift = nn.Linear(2, width)
        self.spec = nn.ModuleList([SpectralConv1d(width, width, modes) for _ in range(n_layers)])
        self.w    = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, x):                                   # x: (batch, n_points, 2)
        x = self.lift(x).permute(0, 2, 1)
        for fourier, local in zip(self.spec, self.w):
            x = nn.functional.gelu(fourier(x) + local(x))
        x = x.permute(0, 2, 1)
        return self.proj(x).squeeze(-1)                     # (batch, n_points)

## 6. Loss: relative L2 error 

In [7]:
def rel_l2(pred, target):
    error_size  = torch.linalg.vector_norm(pred - target, dim=-1)
    target_size = torch.linalg.vector_norm(target, dim=-1)
    return (error_size / target_size).mean()

## 7. Train the final model and save it
500 epochs on all 240 examples, batch size 16, no validation set. The saved weights are from the last epoch.

In [8]:
set_seed(SEED)
model = FNO1d(MODES, WIDTH, N_LAYERS).to(device)
print("learnable parameters:", sum(p.numel() for p in model.parameters()))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_curve = []
for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(X), device=device)           # random example order each epoch
    for start in range(0, len(X), 16):
        idx = order[start:start + 16]
        loss = rel_l2(model(make_input(X[idx])), Y[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        train_curve.append(rel_l2(model(make_input(X)), Y).item())
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}   training error {train_curve[-1]:.5f}")

torch.save({"state": model.state_dict(),
            "config": dict(modes=MODES, width=WIDTH, n_layers=N_LAYERS),
            "seed": SEED, "epochs": EPOCHS, "train_hist": train_curve,
            "data": "all 48 training trajectories"},
           f"fno_modes{MODES}_all48_seed{SEED}.pt")
print(f"saved fno_modes{MODES}_all48_seed{SEED}.pt")

learnable parameters: 1073857
epoch   0   training error 0.55412
epoch  50   training error 0.01868
epoch 100   training error 0.02586
epoch 150   training error 0.02146
epoch 200   training error 0.01922
epoch 250   training error 0.01467
epoch 300   training error 0.01352
epoch 350   training error 0.00585
epoch 400   training error 0.00312
epoch 450   training error 0.00221
epoch 499   training error 0.00204
saved fno_modes64_all48_seed0.pt
